# 5. Can we steal Mason Miller's signs? The glove tell test

**Hypothesis:** before release, Miller's glove looks different on his fastball than on his offspeed pitches, enough that a model can call the pitch from video.

**Tests**
| | Question | Metric |
|---|---|---|
| Supervised | Can glove-only features predict fastball vs offspeed? | game-grouped CV AUC, with a label-permutation p-value |
| Unsupervised | Without labels, do glove-only clusters split along fastball/offspeed? | ARI, NMI, purity of a 2-cluster GMM |
| Control | Does the body *without* the glove do as well? | same metrics |

If the glove beats chance by a wide margin **and** beats the body-only control, that supports a glove tell. Phase analysis says whether it shows up in the set or in the windup.

First, check his actual mix. If changeups are rare, the main test is fastball vs everything else.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))          # so cache/ and reports/ live at the repo root
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pitchscout.workspace import Config, get_pitches, get_clips
from pitchscout import plotting as P

# DEMO=True runs on synthetic data with no network (numbers are NOT real).
DEMO = os.environ.get("PITCHSCOUT_DEMO") == "1"
cfg = Config(pitcher="Mason Miller", start="2024-03-28", end="2025-09-28", n_clips=600, demo=DEMO)
if DEMO:
    print("DEMO MODE: synthetic data, results do not describe a real pitcher")

In [ ]:
from pitchscout import glove, data
df, throws = get_pitches(cfg)
mix = df.pitch_type.value_counts()
pd.DataFrame({"pitches": mix, "share": (mix / mix.sum()).round(3)}).rename(index=P.name)

In [ ]:
clips = get_clips(cfg, df, throws)
idx = clips.glove_pose.index.to_numpy()
y, games = df.loc[idx, "pitch_type"].to_numpy(), df.loc[idx, "game_pk"].to_numpy()
fastballs = set(df.pitch_type) & glove.FASTBALLS
print("fastball codes:", fastballs, "| clips:", pd.Series(y).value_counts().to_dict())

## Main test: fastball vs everything else

In [ ]:
N_PERM = 200   # more permutations = finer p-value (min p = 1/(N_PERM+1))
res = glove.run_tell_test(clips.glove_pose, clips.glove_pix, clips.pose, y, games,
                          positive=fastballs, crops=clips.crops, n_perm=N_PERM, names=data.PITCH_NAMES)
print(res.verdict)
pd.DataFrame(res.sets).T[["auc", "perm_p", "null_auc_95", "accuracy", "majority", "ari", "nmi", "purity"]].round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
s = pd.DataFrame(res.sets).T
ax[0].barh(s.index[::-1], s.auc[::-1], color=[P.SERIES[0] if "Glove" in i else "#898781" for i in s.index[::-1]], height=.55)
ax[0].axvline(.5, color=P.SERIES[1], ls="--", label="coin flip"); ax[0].set_xlim(.4, 1)
ax[0].set_title("Supervised: AUC (fastball vs offspeed)", loc="left"); ax[0].legend(frameon=False)
ax[1].barh(s.index[::-1], s.ari[::-1], color=[P.SERIES[1] if "Glove" in i else "#898781" for i in s.index[::-1]], height=.55)
ax[1].axvline(0, color="#c3c2b7"); ax[1].set_title("Unsupervised: ARI of 2 clusters vs labels", loc="left")
plt.tight_layout(); plt.show()

## When does the glove give it away: in the set or in the windup?

In [ ]:
P.hbar(pd.Series(res.phase_auc).rename({"set": "In the set (before leg lift)", "windup": "Windup (leg lift to cutoff)"}),
       "Glove-only AUC by phase", fmt="{:.3f}"); plt.show()

## Which glove measurements differ most?

In [ ]:
top = pd.Series({glove.label(k): v for k, v in res.top_features})
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(top.index[::-1], top.values[::-1], color=[P.SERIES[0] if v > 0 else P.SERIES[7] for v in top.values[::-1]], height=.6)
ax.axvline(0, color="#c3c2b7"); ax.set_xlabel(f"standardized difference (blue = larger on {res.positive_label})")
ax.set_title("Biggest glove differences", loc="left"); plt.show()

## What the glove looks like

Mean glove crop for each class at six moments (three in the set, three in the windup), and where the two differ.

In [ ]:
if res.mean_crops:
    a, b = res.mean_crops[res.positive_label], res.mean_crops[res.negative_label]
    d = np.abs(a.astype(int) - b.astype(int)).mean(-1)
    fig, ax = plt.subplots(3, len(a), figsize=(2 * len(a), 6.3))
    for j in range(len(a)):
        ax[0, j].imshow(a[j][..., ::-1]); ax[1, j].imshow(b[j][..., ::-1]); ax[2, j].imshow(d[j], cmap=P.BLUES, vmin=0, vmax=d.max())
        ax[0, j].set_title("set" if j < len(a) // 2 else "windup", fontsize=9)
    for i, lab in enumerate([res.positive_label, res.negative_label, "difference"]):
        ax[i, 0].set_ylabel(lab)
    for x in ax.ravel(): x.set_xticks([]); x.set_yticks([])
    plt.tight_layout(); plt.show()

## Was it fixed? Fastball-vs-offspeed AUC by season

If the tell was reported and then fixed, the glove signal should fade in later games.

In [ ]:
season = df.loc[idx, "game_date"].dt.year.to_numpy()
rows = {}
for yr in np.unique(season):
    m = season == yr
    yb = glove.binary_target(y[m], fastballs)
    if m.sum() >= 40 and 0 < yb.mean() < 1:
        rows[yr] = glove.test_feature_set(clips.glove_pose[m], yb, games[m], n_perm=100)
pd.DataFrame(rows).T[["auc", "perm_p", "ari"]].round(3) if rows else "not enough clips per season"

## Fastball vs changeup only (if there are enough changeups)

In [ ]:
ch = {"CH", "FS"} & set(y)
if ch and np.isin(y, list(ch)).sum() >= 25:
    res_ch = glove.run_tell_test(clips.glove_pose, clips.glove_pix, clips.pose, y, games, positive=fastballs,
                                 negative=ch, crops=clips.crops, n_perm=N_PERM, names=data.PITCH_NAMES)
    print(res_ch.verdict)
    display(pd.DataFrame(res_ch.sets).T[["auc", "perm_p", "ari", "nmi", "purity"]].round(3))
else:
    print("Too few changeups in the clip sample for a separate test; increase n_clips or the date range.")

## Save tables for the paper

In [ ]:
from pathlib import Path
if not DEMO:
    Path("reports").mkdir(exist_ok=True)
    pd.DataFrame(res.sets).T.to_csv("reports/glove_tell_test.csv")
    pd.Series(res.phase_auc).to_csv("reports/glove_tell_phase_auc.csv")
    print("saved to reports/")